# Checkpoint D2: Cross-team Testing
Chuẩn bị Skill package để gửi cho nhóm khác.

In [1]:
import json
from pathlib import Path
import shutil

print('Imports THÀNH CÔNG: json, pathlib, shutil')

Imports THÀNH CÔNG: json, pathlib, shutil


In [2]:
# Định nghĩa đường dẫn Skill package của học viên tại outputs
skill_package_path = Path('../../outputs/skills/hr-policy-qa-skill').resolve()

print(f'Đường dẫn Skill package học viên: {skill_package_path}')
print(f'Exists: {skill_package_path.exists()}')

if not skill_package_path.exists():
    print('LỖI: Không tìm thấy thư mục Skill package học viên!')
    print('Vui lòng chạy lại các checkpoint trước để đảm bảo đầy đủ.')


Đường dẫn Skill package học viên: /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill
Exists: True


In [3]:
# Liệt kê tất cả các file trong Skill package và xác minh tính đầy đủ
all_files = []
for f in sorted(skill_package_path.rglob('*')):
    if f.is_file():
        rel_path = f.relative_to(skill_package_path)
        size = f.stat().st_size
        all_files.append({'path': str(rel_path), 'size': size, 'file': f})

# Các file mong đợi (8 file bắt buộc)
required_files = [
    'SKILL.md',
    'skill.json',
    'scripts/chunker.py',
    'scripts/evaluator.py',
    'scripts/retriever.py',
    'schemas/hr-response.schema.json',
    'kb/hr-policies/policy-allowance.md',
    'kb/hr-policies/policy-leave.md',
]

# Đồng thời kiểm tra các file tùy chọn nhưng nên có này
optional_files = [
    'kb/hr-policies/policy-seniority.md',
    'kb/hr-policies/policy-training.md',
    'kb/kb-inventory.md',
]

print(f'Tổng số file tìm thấy: {len(all_files)}')
print()
print(f'{"Đường dẫn file":<45} {"Kích thước":<10} {"Trạng thái"}')
print('=' * 70)

file_paths = {f['path'] for f in all_files}

for f in all_files:
    is_required = f['path'] in required_files
    is_optional = f['path'] in optional_files
    status = 'REQUIRED' if is_required else ('optional' if is_optional else 'extra')
    print(f'{f["path"]:<45} {f["size"]:>8} B  {status}')

# Kiểm tra tính đầy đủ
missing_required = [f for f in required_files if f not in file_paths]
print()
if missing_required:
    print(f'THIẾU các file bắt buộc: {missing_required}')
else:
    print(f'Tất cả {len(required_files)} file bắt buộc đều đầy đủ.')

Tổng số file tìm thấy: 14

Đường dẫn file                                Kích thước Trạng thái
SKILL.md                                         21755 B  REQUIRED
evaluation-report.md                              4707 B  extra
kb/chunks.json                                   17090 B  extra
kb/hr-policies/policy-allowance.md                3010 B  REQUIRED
kb/hr-policies/policy-leave.md                    3296 B  REQUIRED
kb/hr-policies/policy-seniority.md                2741 B  optional
kb/hr-policies/policy-training.md                 4023 B  optional
kb/kb-inventory.md                                1742 B  optional
schemas/hr-response.schema.json                   3519 B  REQUIRED
scripts/__pycache__/retriever.cpython-314.pyc    13901 B  extra
scripts/chunker.py                               11701 B  REQUIRED
scripts/evaluator.py                             18157 B  REQUIRED
scripts/retriever.py                             11207 B  REQUIRED
skill.json                                 

In [4]:
# Xác minh file SKILL.md có đủ 6 phần (sections)
skill_md_path = skill_package_path / 'SKILL.md'

if skill_md_path.exists():
    content = skill_md_path.read_text(encoding='utf-8')
    
    # Đếm tiêu đề level-2 (## Section)
    import re
    sections = re.findall(r'^## (.+)$', content, re.MULTILINE)
    
    print(f'Xác minh SKILL.md:')
    print(f'  Kích thước file: {len(content)} ký tự')
    print(f'  Số phần tìm thấy: {len(sections)}')
    print(f'  Yêu cầu: 6 phần')
    print()
    print('  Các phần:')
    for i, s in enumerate(sections, 1):
        print(f'    {i}. {s}')
    
    skill_md_valid = len(sections) >= 6
    print(f'\n  SKILL.md: {"PASS" if skill_md_valid else "FAIL"} ({len(sections)}/6 phần)')
else:
    skill_md_valid = False
    print('SKILL.md: KHÔNG TÌM THẤY')

Xác minh SKILL.md:
  Kích thước file: 17996 ký tự
  Số phần tìm thấy: 6
  Yêu cầu: 6 phần

  Các phần:
    1. 1. Mô tả & Vai trò (Persona)
    2. 2. Kịch bản kích hoạt (Triggers)
    3. 3. Quy trình thực thi (Execution Workflow)
    4. 4. Định dạng đầu ra (Output Format)
    5. 5. Giới hạn (Boundaries)
    6. 6. Quy tắc an toàn (Safety Rules)

  SKILL.md: PASS (6/6 phần)


In [5]:
# Xác minh file skill.json có đủ các trường yêu cầu
skill_json_path = skill_package_path / 'skill.json'

required_fields = ['name', 'version', 'triggers', 'permissions', 'quality_gates']

if skill_json_path.exists():
    with open(skill_json_path, 'r', encoding='utf-8') as f:
        skill_data = json.load(f)
    
    print('Xác minh skill.json:')
    print(f'  Các key ở level cao nhất: {list(skill_data.keys())}')
    print()
    
    skill_json_valid = True
    for field in required_fields:
        present = field in skill_data
        status = 'PASS' if present else 'FAIL'
        print(f'  {field}: {status}')
        if present:
            val = skill_data[field]
            if isinstance(val, (dict, list)):
                print(f'    Value: {json.dumps(val, ensure_ascii=False)[:100]}')
            else:
                print(f'    Value: {val}')
        if not present:
            skill_json_valid = False
    
    print(f'\n  skill.json: {"PASS" if skill_json_valid else "FAIL"}')
else:
    skill_json_valid = False
    print('skill.json: KHÔNG TÌM THẤY')

Xác minh skill.json:
  Các key ở level cao nhất: ['name', 'version', 'description', 'author', 'created_at', 'triggers', 'permissions', 'required_files', 'quality_gates']

  name: PASS
    Value: hr-policy-qa-skill
  version: PASS
    Value: 1.0.0
  triggers: PASS
    Value: {"keywords": ["nghi phep", "phu cap", "tham nien", "dao tao", "chinh sach", "nhan su", "HR", "nghi",
  permissions: PASS
    Value: {"read_files": ["kb/hr-policies/*.md", "kb/kb-inventory.md"], "write_files": [], "execute_scripts": 
  quality_gates: PASS
    Value: {"min_in_scope_accuracy": ">=75% (6/8)", "out_of_scope_refusal": "100% (2/2)", "citation_rate": ">=9

  skill.json: PASS


In [6]:
# Xác minh schema.json là JSON Schema hợp lệ
schema_path = skill_package_path / 'schemas' / 'hr-response.schema.json'

if schema_path.exists():
    with open(schema_path, 'r', encoding='utf-8') as f:
        schema_data = json.load(f)
    
    print('Xác minh schema.json:')
    print(f'  File: {schema_path.name}')
    
    # Kiểm tra các từ khóa bắt buộc của JSON Schema
    schema_valid = True
    checks = {
        'has $schema': '$schema' in schema_data,
        'has type': 'type' in schema_data,
        'has properties or fields': 'properties' in schema_data or 'fields' in schema_data,
    }
    
    for check, passed in checks.items():
        print(f'  {check}: {"PASS" if passed else "WARN"}')
        if not passed:
            schema_valid = False
    
    # In chi tiết Schema
    print(f'\n  Nội dung Schema:')
    print(f'    $schema: {schema_data.get("$schema", "N/A")}')
    print(f'    type: {schema_data.get("type", "N/A")}')
    if 'properties' in schema_data:
        props = list(schema_data['properties'].keys())
        print(f'    properties: {props}')
    
    print(f'\n  schema.json: {"PASS" if schema_valid else "WARN"}')
else:
    schema_valid = False
    print('schema.json: KHÔNG TÌM THẤY')

Xác minh schema.json:
  File: hr-response.schema.json
  has $schema: PASS
  has type: PASS
  has properties or fields: PASS

  Nội dung Schema:
    $schema: http://json-schema.org/draft-07/schema#
    type: object
    properties: ['question', 'classification', 'answer', 'citations', 'confidence', 'is_out_of_scope', 'refusal_message', 'self_check_result', 'retrieval_method', 'top_chunks_used']

  schema.json: PASS


In [7]:
# Xác minh thư mục scripts/ có đủ 3 file .py
scripts_dir = skill_package_path / 'scripts'

if scripts_dir.exists():
    py_files = list(scripts_dir.glob('*.py'))
    expected_scripts = ['chunker.py', 'evaluator.py', 'retriever.py']
    
    print('Xác minh scripts/:')
    print(f'  Số file Python tìm thấy: {len(py_files)}')
    print(f'  Mong đợi: 3 (chunker.py, evaluator.py, retriever.py)')
    print()
    
    scripts_valid = True
    for expected in expected_scripts:
        exists = (scripts_dir / expected).exists()
        status = 'PASS' if exists else 'FAIL'
        print(f'  {expected}: {status}')
        if exists:
            size = (scripts_dir / expected).stat().st_size
            lines = len((scripts_dir / expected).read_text(encoding='utf-8').splitlines())
            print(f'    Kích thước: {size} bytes, Số dòng: {lines}')
        if not exists:
            scripts_valid = False
    
    print(f'\n  scripts/: {"PASS" if scripts_valid else "FAIL"}')
else:
    scripts_valid = False
    print('scripts/: THƯ MỤC KHÔNG TÌM THẤY')

Xác minh scripts/:
  Số file Python tìm thấy: 3
  Mong đợi: 3 (chunker.py, evaluator.py, retriever.py)

  chunker.py: PASS
    Kích thước: 11701 bytes, Số dòng: 378
  evaluator.py: PASS
    Kích thước: 18157 bytes, Số dòng: 504
  retriever.py: PASS
    Kích thước: 11207 bytes, Số dòng: 375

  scripts/: PASS


In [8]:
# Báo cáo mức độ sẵn sàng kiểm thử chéo
print('=' * 70)
print('BÁO CÁO SẴN SÀNG KIỂM THỬ CHÉO')
print('=' * 70)
print()

# Tóm tắt danh sách file
print(f'Số file trong package:     {len(all_files)}')
print(f'Số file bắt buộc:         {len(required_files) - len(missing_required)}/{len(required_files)}')
print()

# Kết quả xác minh
validations = {
    'SKILL.md (6 phần)': skill_md_valid if 'skill_md_valid' in dir() else False,
    'skill.json (5 trường)': skill_json_valid if 'skill_json_valid' in dir() else False,
    'schema.json (hợp lệ)':   schema_valid if 'schema_valid' in dir() else False,
    'scripts/ (3 .py)':      scripts_valid if 'scripts_valid' in dir() else False,
    'Đủ các file bắt buộc (8)':    len(missing_required) == 0 if 'missing_required' in dir() else False,
}

print('Kết quả xác minh:')
all_valid = True
for name, passed in validations.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  {name:<30} {status}')
    if not passed:
        all_valid = False

print()
ready = 'YES' if all_valid else 'NO'
print(f'Sẵn sàng chia sẻ: {ready}')

if not all_valid:
    print()
    print('Các vấn đề cần khắc phục:')
    for name, passed in validations.items():
        if not passed:
            print(f'  - {name}: cần chú ý')

BÁO CÁO SẴN SÀNG KIỂM THỬ CHÉO

Số file trong package:     14
Số file bắt buộc:         8/8

Kết quả xác minh:
  SKILL.md (6 phần)              PASS
  skill.json (5 trường)          PASS
  schema.json (hợp lệ)           PASS
  scripts/ (3 .py)               PASS
  Đủ các file bắt buộc (8)       PASS

Sẵn sàng chia sẻ: YES


In [9]:
# Tiến hành nén zip Skill package học viên để chia sẻ
zip_output_path = Path('../../outputs/skills/hr-policy-qa-skill')
if skill_package_path.exists() and all_valid:
    print(f"Đang nén zip thư mục {skill_package_path}...")
    shutil.make_archive(str(zip_output_path), 'zip', str(skill_package_path))
    print(f"✓ Đóng gói THÀNH CÔNG: {zip_output_path}.zip")
    print(f"Kích thước: {Path(str(zip_output_path) + '.zip').stat().st_size} bytes")
else:
    print("[LỖI] Không thể đóng gói vì thiếu file hoặc thư mục không tồn tại.")


Đang nén zip thư mục /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-02/session-04-knowledge-base-rag-hr-policy-qa/outputs/skills/hr-policy-qa-skill...
✓ Đóng gói THÀNH CÔNG: ../../outputs/skills/hr-policy-qa-skill.zip
Kích thước: 42285 bytes


In [10]:
print('=' * 60)
print('HƯỚNG DẪN CHIA SẺ VỚI NHÓM KHÁC')
print('=' * 60)
print()
print('Để chia sẻ Skill package với nhóm khác:')
print()
print('1. Nén zip thư mục Skill package:')
print('   cd templates/skills/')
print('   zip -r hr-policy-qa-skill.zip hr-policy-qa-skill/')
print()
print('2. Gửi file zip cho nhóm khác (qua chat, email, hoặc shared drive).')
print()
print('3. Nhóm nhận làm theo hướng dẫn:')
print('   a. Giải nén: unzip hr-policy-qa-skill.zip')
print('   b. Cài đặt các thư viện: pip install -r requirements.txt')
print('   c. Chạy evaluator: python scripts/evaluator.py')
print('   d. Báo cáo kết quả: điền vào biểu mẫu báo cáo')
print()
print('4. Nhóm nhận cần báo cáo:')
print('   - Số câu hỏi đạt/không đạt (pass/fail)')
print('   - Các chỉ số SLI (độ chính xác, tỷ lệ từ chối, tỷ lệ trích dẫn)')
print('   - Các câu hỏi không đạt kèm lý do')
print('   - Góp ý cải thiện')
print()
print('5. Sau khi nhận phản hồi:')
print('   - Cập nhật lại Skill package')
print('   - Chạy lại evaluator')
print('   - Gửi bản cập nhật cho nhóm đánh giá')
print()
print('CHECKPOINT D2 HOÀN THÀNH.')

HƯỚNG DẪN CHIA SẺ VỚI NHÓM KHÁC

Để chia sẻ Skill package với nhóm khác:

1. Nén zip thư mục Skill package:
   cd templates/skills/
   zip -r hr-policy-qa-skill.zip hr-policy-qa-skill/

2. Gửi file zip cho nhóm khác (qua chat, email, hoặc shared drive).

3. Nhóm nhận làm theo hướng dẫn:
   a. Giải nén: unzip hr-policy-qa-skill.zip
   b. Cài đặt các thư viện: pip install -r requirements.txt
   c. Chạy evaluator: python scripts/evaluator.py
   d. Báo cáo kết quả: điền vào biểu mẫu báo cáo

4. Nhóm nhận cần báo cáo:
   - Số câu hỏi đạt/không đạt (pass/fail)
   - Các chỉ số SLI (độ chính xác, tỷ lệ từ chối, tỷ lệ trích dẫn)
   - Các câu hỏi không đạt kèm lý do
   - Góp ý cải thiện

5. Sau khi nhận phản hồi:
   - Cập nhật lại Skill package
   - Chạy lại evaluator
   - Gửi bản cập nhật cho nhóm đánh giá

CHECKPOINT D2 HOÀN THÀNH.
